# HAR elastic net — recency weighting × window length (3D QLIKE surfaces)

Companion to `HAR_RegressionWithElasticNet_WindowSweep.ipynb`. That notebook fit the elastic
net on a **plain, equally-weighted** rolling window. This one adds a **recency-decay factor δ**:
each observation in the window is weighted `δ^age` (newest row age 0 → weight 1; oldest →
`δ^(W-1)`), so `δ < 1` tilts the fit toward recent observations and `δ = 1` recovers the
unweighted base case. Weights are normalised to mean 1 so `alpha`'s penalty scale stays
comparable across δ. (`enet_path` takes no `sample_weight`, so inside the selector the weighting
is applied via weighted-centring + √weight row-scaling — the transform sklearn does internally.)

Tuning is unchanged: at every `(window, δ)` we grid-search `(alpha, l1_ratio)` and keep the pair
that **minimises the overall out-of-sample rolling QLIKE** (deliberate look-ahead → reports the
*best achievable* EN QLIKE, a clean upper bound, not a tradeable rule). Every window is scored on
the **same common out-of-sample period** so only window length and δ change.

The window sweep is trimmed to **1y, 1.5y, 2y, 2.5y, 3y** and δ to **[1.0, 0.999, 0.995, 0.99,
0.97]** (25 fits per spec) to keep the 3D sweep tractable. The result is one **3D QLIKE surface
per spec** — x = window length, y = δ, z = best-EN QLIKE — answering whether any recency tilt
beats the δ=1 (unweighted) edge.

Same three specs as the source notebook (logged HAR + logged exogenous day-t terms, Duan-smearing
log→level back-transform, QLIKE-in-levels scoring):
- **Run 18** — log HAR + log GVZ
- **Run 19** — log HAR + log SPX (RV_ES) + log GVZ
- **Run 20** — log HAR + log crude (RV_crude) + log GVZ

In [ ]:
# ===========================================================================
# Cell 1 — Imports & data
# ===========================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (enables projection="3d")
from sklearn.linear_model import ElasticNet, enet_path
from sklearn.preprocessing import StandardScaler

# Same aligned daily realized-variance panel used by SimpleHAR_Regressions.ipynb
data = pd.read_parquet("merged_RV_GVZ.parquet")
rv = data["RV_gold"].astype(float)

TRADING_DAYS = 252
WINDOW_YEARS = np.array([1.0, 1.5, 2.0, 2.5, 3.0])            # reduced grid for the 3D sweep
WINDOWS = [int(round(yr * TRADING_DAYS)) for yr in WINDOW_YEARS]
WINDOW_DEFAULT = 504          # 2y, used for the sanity-check cell
EPS = 1e-6                    # QLIKE forecast floor

print(f"RV_gold: {len(rv)} obs, {rv.index.min().date()} .. {rv.index.max().date()}")
print(f"Columns available: {list(data.columns)}")
print(f"Window lengths (days): {WINDOWS}")

In [2]:
# ===========================================================================
# Cell 2 — Build the 3 log+GVZ design tables (mirror SimpleHAR Cells 7 & 11)
# ===========================================================================
# Log-HAR components on x = log(RV_gold):
#   x_d[t]=x_t, x_w[t]=mean(x_{t-4..t}), x_m[t]=mean(x_{t-21..t})
#   y_log[t]=x_{t+1}=log(RV_{t+1}); y_level[t]=RV_{t+1} (kept for QLIKE in levels).
# Exogenous day-t terms are logged and known at the close (no look-ahead).

# Strict positivity required for every logged input
for col in ["RV_gold", "GVZ_close", "RV_ES", "RV_crude"]:
    assert (data[col] > 0).all(), f"{col} has non-positive values; log undefined"

x = np.log(rv)

def build_log_design(extra_cols):
    """Base log-HAR design + optional logged exogenous day-t columns (dict name->series)."""
    df = pd.DataFrame(index=rv.index)
    df["x_d"] = x
    df["x_w"] = x.rolling(5).mean()
    df["x_m"] = x.rolling(22).mean()
    for name, series in extra_cols.items():
        df[name] = series.reindex(rv.index)
    df["y_log"]   = x.shift(-1)        # log(RV_{t+1})
    df["y_level"] = rv.shift(-1)       # RV_{t+1} in levels, for QLIKE
    return df.dropna()

log_gvz   = np.log(data["GVZ_close"])
log_spx   = np.log(data["RV_ES"])
log_crude = np.log(data["RV_crude"])

# Run 18 / 19 / 20 design tables
d_gvz       = build_log_design({"log_GVZ": log_gvz})
d_spx_gvz   = build_log_design({"log_GVZ": log_gvz, "log_RV_ES": log_spx})
d_crude_gvz = build_log_design({"log_GVZ": log_gvz, "log_RV_crude": log_crude})

for name, df in [("d_gvz", d_gvz), ("d_spx_gvz", d_spx_gvz), ("d_crude_gvz", d_crude_gvz)]:
    assert df.notna().all().all(), f"unexpected NaNs in {name}"

print("Run 18  log+GVZ        cols:", list(d_gvz.columns),       f"({len(d_gvz)} rows)")
print("Run 19  log+SPX+GVZ    cols:", list(d_spx_gvz.columns),   f"({len(d_spx_gvz)} rows)")
print("Run 20  log+crude+GVZ  cols:", list(d_crude_gvz.columns), f"({len(d_crude_gvz)} rows)")
d_gvz.head()

Run 18  log+GVZ        cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'y_log', 'y_level'] (4092 rows)
Run 19  log+SPX+GVZ    cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'log_RV_ES', 'y_log', 'y_level'] (4092 rows)
Run 20  log+crude+GVZ  cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'log_RV_crude', 'y_log', 'y_level'] (4092 rows)


,x_d,x_w,x_m,log_GVZ,y_log,y_level
Date,,,,,,
2010-02-03,2.850663,2.981362,2.933665,3.095125,3.580491,35.891145
2010-02-04,3.580491,3.080406,2.967351,3.302849,3.444815,31.337489
2010-02-05,3.444815,3.158204,2.994161,3.317453,2.911737,18.388714
2010-02-08,2.911737,3.146146,2.996290,3.291383,2.940123,18.918172
2010-02-09,2.940123,3.145566,3.005753,3.236716,3.123939,22.735769


In [ ]:
# ===========================================================================
# Cell 3 — Helpers: QLIKE, common-OOS start, OLS baseline, elastic-net grid search
# ===========================================================================
# Common out-of-sample period: every window length is scored on the SAME forecast
# dates (those on/after the longest window has warmed up), so differences reflect
# window length only — mirrors the SimpleHAR_Regressions.ipynb window sweep.
START_DATE = d_gvz.index[max(WINDOWS)]

# Elastic-net tuning grid (selected by min out-of-sample QLIKE; no CV).
ALPHAS    = np.logspace(-4, 1, 25)                 # overall penalty grid
# The source window-sweep notebook found l1_ratio* = 0.1 at EVERY window/spec — this
# data sits at the near-ridge end. We keep just {0.1, 0.3} (each l1 is a separate
# path-solve, the dominant cost) so the 5x5 delta x window 3D sweep stays tractable;
# 0.3 is retained as a sanity check that 0.1 still wins under recency weighting.
L1_RATIOS = [0.1, 0.3]                              # L1/L2 mix (0->ridge .. 1->lasso)


def _recency_weights(n, delta):
    """Geometric recency weights for a rolling window of length n.

    Newest row (last in the window) gets weight 1; the row `age` steps older gets
    delta**age. delta=1.0 -> equal weights (the unweighted base case). Weights are
    normalised to mean 1 (sum = n) so they only re-balance the fit *relative* to
    each other and keep the ElasticNet `alpha` penalty scale comparable across delta.
    """
    if delta >= 1.0:
        return np.ones(n)
    ages = np.arange(n)[::-1]          # oldest = n-1 ... newest = 0
    w = delta ** ages
    return w * (n / w.sum())           # mean-1 normalise

def _qlike(actual, forecast, eps=EPS):
    f = np.maximum(forecast, eps)
    r = actual / f
    return r - np.log(r) - 1.0, int((forecast <= eps).sum())

# --- OLS log+smearing baseline, common-OOS gated (mirror SimpleHAR) ---
def rolling_log_qlike_smearing_eval(design, feat_cols, window, start_date=None):
    if start_date is None:
        start_date = START_DATE
    X = np.column_stack([np.ones(len(design)), design[feat_cols].to_numpy()])
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    fc, ac = [], []
    for t in range(window, len(design)):
        if idx[t] < start_date:
            continue
        Xw, yw = X[t - window:t], yl[t - window:t]
        beta, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
        smearing = np.mean(np.exp(yw - Xw @ beta))          # Duan factor
        fc.append(np.exp(X[t] @ beta) * smearing); ac.append(lvl[t])
    q, clip = _qlike(np.array(ac), np.array(fc))
    return q.mean(), len(q), clip

# --- Elastic net at a FIXED (alpha, l1_ratio); optional recency weighting (delta) ---
def rolling_log_elasticnet_eval(design, feat_cols, alpha, l1_ratio,
                                window, start_date=None, delta=1.0):
    if start_date is None:
        start_date = START_DATE
    Xf  = design[feat_cols].to_numpy()
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    w = _recency_weights(window, delta)              # per-observation sample weights
    fc, ac = [], []
    for t in range(window, len(design)):
        if idx[t] < start_date:
            continue
        Xw, yw = Xf[t - window:t], yl[t - window:t]
        sc = StandardScaler().fit(Xw)
        Xw_s = sc.transform(Xw)
        m = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, fit_intercept=True,
                       max_iter=5000).fit(Xw_s, yw, sample_weight=w)
        x_hat = m.predict(sc.transform(Xf[t:t + 1]))[0]
        smearing = np.average(np.exp(yw - m.predict(Xw_s)), weights=w)   # weighted Duan factor
        fc.append(np.exp(x_hat) * smearing); ac.append(lvl[t])
    q, clip = _qlike(np.array(ac), np.array(fc))
    return q.mean(), len(q), clip

# --- Grid search: pick (alpha, l1_ratio) minimising the common-OOS QLIKE ---
# Efficiency: enet_path solves the WHOLE alpha grid for a given l1_ratio in one
# coordinate-descent pass (warm starts), so cost is n_OOS * n_l1 path-solves, not
# n_OOS * n_alpha individual fits.
# Recency weighting (delta<1): enet_path has no sample_weight, so we reproduce a
# weighted fit by (a) weighted-centring X and y per window and (b) scaling each row
# by sqrt(w) — the exact transform sklearn applies internally. At delta=1.0 w=ones,
# the weighted means collapse to plain means (X already standardised -> mean ~0),
# so this reduces to the original unweighted path.
def select_best_elasticnet(design, feat_cols, window, start_date=None,
                           alphas=ALPHAS, l1_ratios=L1_RATIOS, delta=1.0):
    if start_date is None:
        start_date = START_DATE
    Xf  = design[feat_cols].to_numpy()
    yl  = design["y_log"].to_numpy()
    lvl = design["y_level"].to_numpy()
    idx = design.index
    w  = _recency_weights(window, delta)
    sw = np.sqrt(w)
    ts = [t for t in range(window, len(design)) if idx[t] >= start_date]
    n = len(ts)
    n_a = len(alphas)
    qsum = {l1: np.zeros(n_a) for l1 in l1_ratios}
    alpha_grid = None
    for t in ts:
        Xw, yw = Xf[t - window:t], yl[t - window:t]
        sc = StandardScaler().fit(Xw)
        Xw_s = sc.transform(Xw)
        x_new = sc.transform(Xf[t:t + 1])           # (1, p)
        Xc_mean = np.average(Xw_s, axis=0, weights=w)   # weighted column means (~0 if delta=1)
        Xc = Xw_s - Xc_mean
        ymean = np.average(yw, weights=w)
        yc = yw - ymean
        actual = lvl[t]
        for l1 in l1_ratios:
            a_out, coefs, _ = enet_path(Xc * sw[:, None], yc * sw,
                                        l1_ratio=l1, alphas=alphas, max_iter=5000)
            # coefs: (p, n_a), aligned to a_out — evaluate in the UNSCALED space
            fitted = Xc @ coefs + ymean                                 # (win, n_a)
            smear = np.average(np.exp(yw[:, None] - fitted),
                               axis=0, weights=w)                        # (n_a,)
            x_hat = ((x_new - Xc_mean) @ coefs)[0] + ymean              # (n_a,)
            fc = np.maximum(np.exp(x_hat) * smear, EPS)                 # (n_a,)
            r = actual / fc
            qsum[l1] += r - np.log(r) - 1.0
            alpha_grid = a_out
    # mean QLIKE per (l1, alpha); locate the global minimum
    best_q, best_alpha, best_l1 = np.inf, None, None
    for l1 in l1_ratios:
        qmean = qsum[l1] / n
        j = int(np.argmin(qmean))
        if qmean[j] < best_q:
            best_q, best_alpha, best_l1 = qmean[j], float(alpha_grid[j]), l1
    return best_q, best_alpha, best_l1, n

print(f"Common OOS start: {START_DATE.date()}  "
      f"({int((d_gvz.index >= START_DATE).sum())} forecast days)")
print(f"Grid: {len(ALPHAS)} alphas x {len(L1_RATIOS)} l1_ratios; windows={WINDOWS}")

In [4]:
# ===========================================================================
# Cell 4 — Sanity check at the default 2-year window
# ===========================================================================
specs = [
    ("Run 18  log+GVZ",       d_gvz,       ["x_d", "x_w", "x_m", "log_GVZ"]),
    ("Run 19  log+SPX+GVZ",   d_spx_gvz,   ["x_d", "x_w", "x_m", "log_GVZ", "log_RV_ES"]),
    ("Run 20  log+crude+GVZ", d_crude_gvz, ["x_d", "x_w", "x_m", "log_GVZ", "log_RV_crude"]),
]

print(f"Default window = {WINDOW_DEFAULT} days (~{WINDOW_DEFAULT/TRADING_DAYS:.1f}y), "
      f"common OOS from {START_DATE.date()}\n")
for label, design, feats in specs:
    q_ols, n_ols, _ = rolling_log_qlike_smearing_eval(design, feats, WINDOW_DEFAULT)
    q_en, a_star, l1_star, n_en = select_best_elasticnet(design, feats, WINDOW_DEFAULT)
    # cross-check: refit a plain ElasticNet at the selected (alpha*, l1*)
    q_chk, _, _ = rolling_log_elasticnet_eval(design, feats, a_star, l1_star, WINDOW_DEFAULT)
    print(f"{label:<24} OLS {q_ols:.6f} | EN* {q_en:.6f} "
          f"(alpha*={a_star:.4g}, l1*={l1_star:.2f}, refit-check {q_chk:.6f})  "
          f"delta {q_en - q_ols:+.6f}  [n={n_en}]")

Default window = 504 days (~2.0y), common OOS from 2015-02-18

Run 18  log+GVZ          OLS 0.028915 | EN* 0.028915 (alpha*=0.0006813, l1*=0.10, refit-check 0.028915)  delta -0.000000  [n=2832]
Run 19  log+SPX+GVZ      OLS 0.028990 | EN* 0.028985 (alpha*=0.002873, l1*=0.10, refit-check 0.028985)  delta -0.000004  [n=2832]
Run 20  log+crude+GVZ    OLS 0.028835 | EN* 0.028835 (alpha*=0.0006813, l1*=0.10, refit-check 0.028835)  delta -0.000000  [n=2832]


In [5]:
# ===========================================================================
# Cell 5 — Window-length sweep (OLS vs best elastic net), common OOS
# ===========================================================================
ols_by_window = pd.DataFrame(index=WINDOW_YEARS, columns=[s[0] for s in specs], dtype=float)
en_by_window  = pd.DataFrame(index=WINDOW_YEARS, columns=[s[0] for s in specs], dtype=float)
alpha_sel     = pd.DataFrame(index=WINDOW_YEARS, columns=[s[0] for s in specs], dtype=float)
l1_sel        = pd.DataFrame(index=WINDOW_YEARS, columns=[s[0] for s in specs], dtype=float)

for label, design, feats in specs:
    for yr, w in zip(WINDOW_YEARS, WINDOWS):
        q_ols, _, _ = rolling_log_qlike_smearing_eval(design, feats, w)
        q_en, a_star, l1_star, _ = select_best_elasticnet(design, feats, w)
        ols_by_window.loc[yr, label] = q_ols
        en_by_window.loc[yr, label]  = q_en
        alpha_sel.loc[yr, label]     = a_star
        l1_sel.loc[yr, label]        = l1_star
    print(f"{label}: done")

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
print("\nOLS QLIKE by window (years x spec):")
print(ols_by_window.to_string())
print("\nBest elastic-net QLIKE by window (years x spec):")
print(en_by_window.to_string())
print("\nSelected alpha* by window:")
print(alpha_sel.to_string())
print("\nSelected l1_ratio* by window:")
print(l1_sel.to_string())

print("\nBest window per spec (lower QLIKE = better):")
for label, *_ in specs:
    w_ols = ols_by_window[label].idxmin(); q_ols = ols_by_window.loc[w_ols, label]
    w_en  = en_by_window[label].idxmin();  q_en  = en_by_window.loc[w_en, label]
    flag = "EN beats OLS" if q_en < q_ols else f"EN +{q_en - q_ols:.6f} vs OLS"
    print(f"  {label:<24} OLS best {w_ols:.1f}y={q_ols:.6f} | "
          f"EN best {w_en:.1f}y={q_en:.6f}  ({flag})")

Run 18  log+GVZ: done
Run 19  log+SPX+GVZ: done
Run 20  log+crude+GVZ: done

OLS QLIKE by window (years x spec):
          Run 18  log+GVZ  Run 19  log+SPX+GVZ  Run 20  log+crude+GVZ
1.000000         0.028743             0.028890               0.028799
1.500000         0.028799             0.028905               0.028731
2.000000         0.028915             0.028990               0.028835
2.500000         0.029022             0.029079               0.028986
3.000000         0.029019             0.029071               0.029014
3.500000         0.028924             0.028918               0.028878
4.000000         0.028855             0.028834               0.028776
4.500000         0.028818             0.028818               0.028810
5.000000         0.028822             0.028821               0.028778

Best elastic-net QLIKE by window (years x spec):
          Run 18  log+GVZ  Run 19  log+SPX+GVZ  Run 20  log+crude+GVZ
1.000000         0.028705             0.028841               0.0287

In [ ]:
# ===========================================================================
# Cell 6 — Recency-decay (delta) x window sweep: best elastic-net QLIKE cube
# ===========================================================================
# For each spec, sweep the rolling window length AND the recency-decay factor delta.
# At every (window, delta) we still grid-search (alpha, l1_ratio) and keep the min-
# QLIKE pair, so each surface cell shows the *best achievable* EN QLIKE at that delta.
#   delta = 1.0  -> equal weighting (matches the unweighted en_by_window column)
#   delta < 1.0  -> geometric down-weighting of older observations
# 5 windows x 5 deltas = 25 selector calls per spec (75 total).
DELTAS = [1.0, 0.999, 0.995, 0.99, 0.97]

# best-EN QLIKE cube: en_cube[label] is a DataFrame (index=WINDOW_YEARS, cols=DELTAS)
en_cube = {label: pd.DataFrame(index=WINDOW_YEARS, columns=DELTAS, dtype=float)
           for label, *_ in specs}

for label, design, feats in specs:
    for yr, w in zip(WINDOW_YEARS, WINDOWS):
        for delta in DELTAS:
            q_en, _, _, _ = select_best_elasticnet(design, feats, w, delta=delta)
            en_cube[label].loc[yr, delta] = q_en
    print(f"{label}: done")

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
for label, *_ in specs:
    print(f"\nBest EN QLIKE — {label}  (rows=window yrs, cols=delta):")
    print(en_cube[label].to_string())
    df = en_cube[label]
    yr_star = df.min(axis=1).idxmin()
    d_star  = df.loc[yr_star].idxmin()
    q_star  = df.loc[yr_star, d_star]
    q_base  = df.loc[yr_star, 1.0]
    print(f"  min QLIKE {q_star:.6f} at window={yr_star:.1f}y, delta={d_star} "
          f"(vs delta=1.0 at same window {q_base:.6f}, "
          f"{'improves' if q_star < q_base else 'no gain'} {q_star - q_base:+.6f})")

In [ ]:
# ===========================================================================
# Cell 7 — 3D QLIKE surfaces: window x delta -> best elastic-net QLIKE (per spec)
# ===========================================================================
# One surface per spec. x = rolling-window length, y = recency-decay delta, and the
# vertical z = average out-of-sample QLIKE (lower = better). delta is plotted on an
# evenly-spaced axis (its values bunch near 1.0) with tick labels showing the actual
# factor. The black marker is the (window, delta) cell with the lowest QLIKE.
ypos = np.arange(len(DELTAS))                         # even spacing for the delta axis
WIN, DELi = np.meshgrid(WINDOW_YEARS, ypos)          # (n_delta, n_window)

def plot_en_surface(label, df, fname):
    Z = df.to_numpy(dtype=float).T                   # (n_delta, n_window)
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(projection="3d")
    surf = ax.plot_surface(WIN, DELi, Z, cmap="viridis", edgecolor="k",
                           linewidth=0.3, alpha=0.9, antialiased=True)
    # mark the global minimum on the surface
    yr_star = df.min(axis=1).idxmin()
    d_star  = df.loc[yr_star].idxmin()
    q_star  = df.loc[yr_star, d_star]
    ax.scatter([yr_star], [DELTAS.index(d_star)], [q_star], color="red", s=60,
               depthshade=False, label=f"min @ {yr_star:.1f}y, δ={d_star}")
    ax.set_xlabel("Rolling-window length (years)")
    ax.set_ylabel("Recency decay δ")
    ax.set_zlabel("Avg OOS QLIKE")
    ax.set_yticks(ypos); ax.set_yticklabels([f"{d:g}" for d in DELTAS])
    ax.set_title(f"Best elastic-net QLIKE surface — {label}\n"
                 "(z = min-QLIKE over alpha,l1 at each window × δ)")
    ax.view_init(elev=22, azim=-60)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=12, pad=0.1, label="QLIKE")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.show()

fnames = {
    "Run 18  log+GVZ":       "qlike_en_surface_run18.png",
    "Run 19  log+SPX+GVZ":   "qlike_en_surface_run19.png",
    "Run 20  log+crude+GVZ": "qlike_en_surface_run20.png",
}
for label, *_ in specs:
    plot_en_surface(label, en_cube[label], fnames[label])